In [ ]:
# Example 1: Import the OpenTelemetrySpan class -- the primary API surface.
from azure.core.tracing.ext.opentelemetry_span import OpenTelemetrySpan

print(f"OpenTelemetrySpan class: {OpenTelemetrySpan}")
print(f"Module: {OpenTelemetrySpan.__module__}")

# Sanity check that the expected methods are present on the class.
expected_methods = ['span', 'kind', 'start', 'finish', 'to_header', 'link', 'add_attribute']
for method_name in expected_methods:
    assert hasattr(OpenTelemetrySpan, method_name), f"OpenTelemetrySpan missing {method_name}"
    print(f"  has: {method_name}")

assert OpenTelemetrySpan is not None

In [ ]:
# Example 2: Create an OpenTelemetrySpan wrapping a real OpenTelemetry tracer span.
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from azure.core.tracing.ext.opentelemetry_span import OpenTelemetrySpan

# Set up a basic OTel tracer provider (in-memory, no exporter).
provider = TracerProvider()
trace.set_tracer_provider(provider)
tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span("test-parent") as otel_span:
    wrapped = OpenTelemetrySpan(span=otel_span, name="azure-wrapped")
    print(f"wrapped: {wrapped}")
    print(f"wrapped.span_instance = {wrapped.span_instance}")
    assert wrapped.span_instance is otel_span

In [ ]:
# Example 3: Exercise the span kind enum + setter.
from opentelemetry import trace
from azure.core.tracing import SpanKind
from azure.core.tracing.ext.opentelemetry_span import OpenTelemetrySpan

tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span("kind-demo") as otel_span:
    wrapped = OpenTelemetrySpan(span=otel_span)
    # SpanKind enum values from azure.core.tracing should be assignable.
    for kind in (SpanKind.CLIENT, SpanKind.SERVER, SpanKind.PRODUCER, SpanKind.CONSUMER, SpanKind.INTERNAL):
        wrapped.kind = kind
        print(f"  set kind = {kind} -> wrapped.kind = {wrapped.kind}")
        assert wrapped.kind == kind

In [ ]:
# Example 4: Round-trip trace context via to_header() / link_from_headers().
from opentelemetry import trace
from azure.core.tracing.ext.opentelemetry_span import OpenTelemetrySpan

tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span("propagator") as otel_span:
    wrapped = OpenTelemetrySpan(span=otel_span)
    headers = wrapped.to_header()
    print(f"propagation headers: {headers}")
    # W3C traceparent must be present and well-formed.
    assert isinstance(headers, dict)
    assert "traceparent" in headers
    parts = headers["traceparent"].split("-")
    assert len(parts) == 4, f"unexpected traceparent shape: {headers['traceparent']}"
    print(f"traceparent ok: version={parts[0]} trace_id={parts[1]}")

print("\nAll azure_core_tracing_opentelemetry smoke tests passed.")